<a href="https://colab.research.google.com/github/AbhiYewale96/AI-Diet-Recovery-Planner/blob/main/Final_Project_Execution_py.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Implement Preprocessing Pipeline:**
load_data(),
handle_missing_values(),
remove_duplicates(),
validate_data(),
verify_relationships(),
feature_engineering(),
save_processed_data().

**Load the datasets**

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving final_cleaned_tracking (1).csv to final_cleaned_tracking (1) (1).csv
Saving final_cleaned_patients (3).csv to final_cleaned_patients (3) (1).csv


**Importing Library**

In [ ]:
import pandas as pd
import numpy as np

**Read the Datasets**

In [ ]:
df = pd.read_csv("/content/final_cleaned_patients (3).csv")


In [ ]:
df1 = pd.read_csv("/content/final_cleaned_tracking (1).csv")

**Handle Missing Values**

In [ ]:
df.isnull().sum()

,0
patient_id,0
name,0
age,0
gender,0
height_cm,0
weight_kg,0
medical_condition,0
disease_type,0
surgery_type,0
medical_history,0


In [ ]:
df1.isnull().sum()

,0
record_id,0
patient_id,0
date,0
weight_kg,0
meals_taken,0
meal_compliance,0
calorie_intake,0
protein_intake,0
carb_intake,0
fat_intake,0


**Validate Data Quality**

**Check Age**

In [ ]:
df['age'].describe()

,age
count,1300.000000
mean,46.780000
std,14.762922
min,18.000000
25%,36.000000
50%,46.000000
75%,58.000000
max,89.000000


**Check BMI**

In [ ]:
df['BMI'].describe()

,BMI
count,1300.000000
mean,26.293014
std,5.471025
min,14.302382
25%,22.347782
50%,26.032007
75%,29.732872
max,44.895376


**Check Blood Sugar Level**

In [ ]:
df1['blood_sugar_level'].describe()

,blood_sugar_level
count,48121.000000
mean,106.033993
std,20.737354
min,65.000000
25%,91.000000
50%,102.900000
75%,118.500000
max,313.000000


**Check Duplicate Records of patients.csv**

In [ ]:
df.duplicated().sum()

np.int64(0)

**Check Duplicate Records of tracking.csv**

In [ ]:
df1.duplicated().sum()

np.int64(0)

**Verify Patient-Tracking Relationship**

In [ ]:
invalid_ids = set(df1['patient_id']) - set(df['patient_id'])
print("Invalid Patient IDs:", len(invalid_ids))

**Create Engineered Features**

**Feature 1: BMI Category**

In [ ]:
def bmi_category(bmi):
    if bmi < 18.5:
        return "Underweight"
    elif bmi < 25:
        return "Normal"
    elif bmi < 30:
        return "Overweight"
    else:
        return "Obese"

df['BMI_Category'] = df['BMI'].apply(bmi_category)

**Feature 2: Age Group**

In [ ]:
df['Age_Group'] = pd.cut(
    df['age'],
    bins=[0,18,35,50,65,100],
    labels=[
        'Youth',
        'Young Adult',
        'Adult',
        'Senior',
        'Elderly'
    ]
)

**Feature 3: Average Blood Sugar**

In [ ]:
avg_sugar = df1.groupby('patient_id')['blood_sugar_level'].mean()

df = df.merge(
    avg_sugar.rename('Avg_Blood_Sugar'),
    on='patient_id',
    how='left'
)

**`Feature 4: Average Steps`**

In [ ]:
avg_steps = df1.groupby('patient_id')['steps_count'].mean()

df = df.merge(
    avg_steps.rename('Avg_Daily_Steps'),
    on='patient_id',
    how='left'
)

**Feature 4: Average Sleep Hours**

In [ ]:
avg_sleep = df1.groupby('patient_id')['sleep_hours'].mean()

df =df.merge(
    avg_sleep.rename('Avg_Sleep_Hours'),
    on='patient_id',
    how='left'
)

**Encoding**

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

df['gender'] = le.fit_transform(df['gender'])
df['disease_type'] = le.fit_transform(df['disease_type'])
df['activity_level'] = le.fit_transform(df['activity_level'])

**Scaling:**
Doctors and reviewers understand Age = 45, but Age_Scaled = -0.72 has no clinical meaning.
The AI/ML team can apply scaling during model training if needed.
If they use Random Forest, XGBoost, or Decision Trees, scaling is not required at all.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

df['Age_Scaled'] = scaler.fit_transform(
    df[['age']]
)

df['BMI_Scaled'] = scaler.fit_transform(
    df[['BMI']]
)

df['Height_cm_Scaled'] = scaler.fit_transform(
    df[['height_cm']]
)

df['Weight_kg_Scaled'] = scaler.fit_transform(
    df[['weight_kg']]
)

**Verify the Data**

In [ ]:
df.head()

,patient_id,name,age,gender,height_cm,weight_kg,medical_condition,disease_type,surgery_type,medical_history,...,alcohol_consumption,existing_conditions,doctor_notes,registration_date,BMI,BMI_Category,Age_Group,Avg_Blood_Sugar,Avg_Daily_Steps,Avg_Sleep_Hours
0,Pat0001,Vivaan White,45,1,175,89.5,Chronic Kidney Disease,9,Kidney Transplant,Underwent Treatment For Chronic Kidney Disease...,...,Not Specified,Pre-Diabetes,Stable condition. Annual review scheduled.,2025-05-09,29.224490,Overweight,Adult,94.095122,2905.524390,7.073171
1,Pat0002,Sanjay Smith,44,1,184,84.1,Chronic Kidney Disease,9,Dialysis Access Surgery,Diagnosed 12 Years Ago,...,Occasional,"Varicose Veins, Sleep Apnea",Patient responding well to current treatment p...,2023-09-13,24.840501,Normal,Adult,93.026230,3093.368852,7.222951
2,Pat0003,Myra Rahman,59,0,171,69.2,Copd,10,No Surgery,Secondary Complications Observed,...,Moderate,No Known Conditions,Blood work shows improvement. Continue current...,2025-01-28,23.665401,Normal,Senior,99.816279,5202.715116,6.331395
3,Pat0004,Riya Wilson,41,0,163,62.9,Chronic Kidney Disease,9,Dialysis Access Surgery,Secondary Complications Observed,...,Occasional,"Pre-Diabetes, Iron Deficiency",Patient responding well to current treatment p...,2025-04-01,23.674207,Normal,Adult,93.020455,12130.352273,7.181818
4,Pat0005,Aadhya Desai,19,0,161,45.0,Anemia,3,No Surgery,Multiple Episodes Requiring Emergency Care,...,Heavy,Varicose Veins,Patient non-compliant with medication schedule...,2024-03-25,17.360441,Underweight,Young Adult,94.812308,13464.946154,7.095385


In [ ]:
df.tail()

,patient_id,name,age,gender,height_cm,weight_kg,medical_condition,disease_type,surgery_type,medical_history,...,alcohol_consumption,existing_conditions,doctor_notes,registration_date,BMI,BMI_Category,Age_Group,Avg_Blood_Sugar,Avg_Daily_Steps,Avg_Sleep_Hours
1295,Pat1296,Mary Riley,37,1,171,116.7,Obesity,5,Gastric Bypass,Post-Surgical Recovery - Stable,...,Moderate,"Allergic Rhinitis, Vitamin D Deficiency",Stable condition. Annual review scheduled.,2025-07-01,39.909716,Obese,Adult,112.060000,9334.866667,7.183333
1296,Pat1297,Sandra Andrews,43,0,169,53.1,Hypertension,0,No Surgery,Chronic Condition Managed With Medication,...,Not Specified,"Osteoporosis, Allergic Rhinitis",Patient non-compliant with medication schedule...,2026-04-18,18.591786,Normal,Adult,113.263333,8567.500000,7.046667
1297,Pat1298,Michelle Roy,26,1,173,76.1,Copd,10,No Surgery,Controlled With Lifestyle Modifications,...,Occasional,Osteoporosis,Referred to specialist for further evaluation.,2024-08-27,25.426844,Overweight,Young Adult,113.093333,7940.000000,6.770000
1298,Pat1299,Rachel Mitchell,76,0,150,79.8,Copd,10,No Surgery,Diagnosed 2 Years Ago,...,Not Specified,No Known Conditions,Patient responding well to current treatment p...,2024-11-25,35.466667,Obese,Elderly,106.330000,7661.233333,6.510000
1299,Pat1300,Katherine Ashley,67,0,153,34.7,Heart Disease,0,Bypass Surgery,Underwent Treatment For Heart Disease In 2023,...,Not Specified,"Fatty Liver, Acid Reflux",Advised dietary modifications. Follow-up in 2 ...,2024-06-28,14.823359,Underweight,Elderly,110.410000,9511.900000,7.323333


In [ ]:
df.shape

(1300, 27)

In [ ]:
df1.shape

(48121, 26)

**Save Processed Datasets After all Tasks**

In [ ]:
df.to_csv("processed_patients.csv", index=False)
df1.to_csv("processed_tracking.csv", index=False)

**Download the Datasets**

In [ ]:
files.download("processed_patients.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
files.download("processed_tracking.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>